In [1]:
# ==============================

# IMPORT LIBRARIES & LOAD DATA

# ==============================

import pandas as pd

import numpy as np

import re

import joblib

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from scipy.sparse import hstack, csr_matrix

from xgboost import XGBRegressor

print("Libraries imported successfully!")

# Load dataset

df = pd.read_csv("vehicle_service_resource_prediction.csv")

print("Dataset Shape:", df.shape)

display(df.head())

print("\nColumns:")

print(df.columns.tolist())

print("\nMissing Values:")

print(df.isnull().sum())

Libraries imported successfully!
Dataset Shape: (50000, 29)


,Service_Record_ID,Vehicle_Type,Vehicle_Age_Years,Mileage_KM,Engine_Type,Service_Category,Customer_Complaint,Vehicle_Symptoms,Service_History_Text,Technician_Notes,...,Day_of_Week,Weekend_Indicator,Customer_Segment,Workshop_Zone,Local_Demand_Index,Appointment_Backlog,Service_Demand,Workshop_Load_Forecast,Technicians_Required,Service_Bays_Required
0,1,MUV,4.7,58925,Hybrid,Brake,brakes feel weak and stopping takes longer,soft pedal and reduced braking,regular maintenance completed 4 times with 2 p...,inspection suggests brake component wear; furt...,...,Thursday,0,Regular,Rural,83.8,19.0,81,95.8,10,16
1,2,Pickup,3.2,37648,Hybrid,AC,air conditioner is not cooling properly,AC noise with poor cooling,regular maintenance completed 1 times with 1 p...,inspection suggests AC system performance; fur...,...,Wednesday,0,Regular,Rural,48.4,11.0,49,62.9,6,10
2,3,SUV,4.9,49835,Electric,General,vehicle requires a general inspection,multiple small maintenance needs,regular maintenance completed 2 times with 4 p...,inspection suggests routine maintenance; furth...,...,Monday,0,Regular,Urban,47.2,14.0,50,71.6,4,10
3,4,Pickup,4.1,30344,Diesel,General,customer reports several minor issues,general wear and inspection required,regular maintenance completed 0 times with 2 p...,inspection suggests routine maintenance; furth...,...,Saturday,1,New,Suburban,40.9,9.0,50,57.4,4,10
4,5,Pickup,9.6,97324,Petrol,Tyre,tyres show uneven wear,road noise and uneven grip,regular maintenance completed 5 times with 6 p...,inspection suggests tyre wear; further checkin...,...,Saturday,1,Regular,Urban,53.4,5.0,66,82.5,7,13



Columns:
['Service_Record_ID', 'Vehicle_Type', 'Vehicle_Age_Years', 'Mileage_KM', 'Engine_Type', 'Service_Category', 'Customer_Complaint', 'Vehicle_Symptoms', 'Service_History_Text', 'Technician_Notes', 'Customer_Concern', 'Previous_Service_Count', 'Previous_Repair_Count', 'Average_Service_Gap_Days', 'Parts_Availability', 'Technician_Availability', 'Workshop_Load', 'Bay_Utilization', 'Season', 'Day_of_Week', 'Weekend_Indicator', 'Customer_Segment', 'Workshop_Zone', 'Local_Demand_Index', 'Appointment_Backlog', 'Service_Demand', 'Workshop_Load_Forecast', 'Technicians_Required', 'Service_Bays_Required']

Missing Values:
Service_Record_ID             0
Vehicle_Type                  0
Vehicle_Age_Years             0
Mileage_KM                    0
Engine_Type                   0
Service_Category              0
Customer_Complaint          150
Vehicle_Symptoms            150
Service_History_Text          0
Technician_Notes            150
Customer_Concern              0
Previous_Service_Count

In [2]:
# ==============================

# DATA CLEANING + NLP PROCESSING

# ==============================

text_features = [

    "Customer_Complaint",

    "Vehicle_Symptoms",

    "Service_History_Text",

    "Technician_Notes",

    "Customer_Concern"

]

categorical_features = [

    "Vehicle_Type",

    "Engine_Type",

    "Service_Category",

    "Season",

    "Day_of_Week",

    "Customer_Segment",

    "Workshop_Zone"

]

numeric_features = [

    "Vehicle_Age_Years",

    "Mileage_KM",

    "Previous_Service_Count",

    "Previous_Repair_Count",

    "Average_Service_Gap_Days",

    "Parts_Availability",

    "Technician_Availability",

    "Workshop_Load",

    "Bay_Utilization",

    "Weekend_Indicator",

    "Local_Demand_Index",

    "Appointment_Backlog"

]

targets = [

    "Service_Demand",

    "Workshop_Load_Forecast",

    "Technicians_Required",

    "Service_Bays_Required"

]

# Fill missing values

for col in text_features:

    df[col] = df[col].fillna("").astype(str)

for col in categorical_features:

    df[col] = df[col].fillna(df[col].mode()[0])

for col in numeric_features:

    df[col] = df[col].fillna(df[col].median())

# Basic text cleaning

def clean_text(text):

    text = text.lower()

    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

for col in text_features:

    df[col] = df[col].apply(clean_text)

print("Data cleaning completed!")

print("\nText Features:")

print(text_features)

print("\nStructured Features:")

print(categorical_features + numeric_features)

Data cleaning completed!

Text Features:
['Customer_Complaint', 'Vehicle_Symptoms', 'Service_History_Text', 'Technician_Notes', 'Customer_Concern']

Structured Features:
['Vehicle_Type', 'Engine_Type', 'Service_Category', 'Season', 'Day_of_Week', 'Customer_Segment', 'Workshop_Zone', 'Vehicle_Age_Years', 'Mileage_KM', 'Previous_Service_Count', 'Previous_Repair_Count', 'Average_Service_Gap_Days', 'Parts_Availability', 'Technician_Availability', 'Workshop_Load', 'Bay_Utilization', 'Weekend_Indicator', 'Local_Demand_Index', 'Appointment_Backlog']


In [3]:
# ==============================

# TF-IDF + ENCODING + SPLIT

# ==============================

X = df[text_features + categorical_features + numeric_features]

y = df[targets]

# Split BEFORE fitting TF-IDF/Encoder

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=42

)

print("Training rows:", X_train.shape[0])

print("Testing rows :", X_test.shape[0])

# --------------------------------

# TF-IDF for 5 NLP features

# --------------------------------

vectorizers = {}

train_text_matrices = []

test_text_matrices = []

for col in text_features:

    vectorizer = TfidfVectorizer(

        max_features=1000,

        ngram_range=(1, 2),

        min_df=2

    )

    train_text = vectorizer.fit_transform(X_train[col])

    test_text = vectorizer.transform(X_test[col])

    vectorizers[col] = vectorizer

    train_text_matrices.append(train_text)

    test_text_matrices.append(test_text)

# Combine all NLP features

X_train_text = hstack(train_text_matrices)

X_test_text = hstack(test_text_matrices)

print("Combined NLP feature shape:", X_train_text.shape)

# --------------------------------

# One-Hot Encoding

# --------------------------------

encoder = OneHotEncoder(

    handle_unknown="ignore",

    sparse_output=True

)

X_train_cat = encoder.fit_transform(X_train[categorical_features])

X_test_cat = encoder.transform(X_test[categorical_features])

# --------------------------------

# Numeric features

# --------------------------------

X_train_num = csr_matrix(

    X_train[numeric_features].values

)

X_test_num = csr_matrix(

    X_test[numeric_features].values

)

# --------------------------------

# Final Feature Matrix

# --------------------------------

X_train_final = hstack([

    X_train_text,

    X_train_cat,

    X_train_num

]).tocsr()

X_test_final = hstack([

    X_test_text,

    X_test_cat,

    X_test_num

]).tocsr()

print("Final training shape:", X_train_final.shape)

print("Final testing shape :", X_test_final.shape)

Training rows: 40000
Testing rows : 10000
Combined NLP feature shape: (40000, 517)
Final training shape: (40000, 564)
Final testing shape : (10000, 564)


In [4]:
# ==============================

# TRAIN 4 XGBOOST MODELS

# ==============================

models = {}

results = {}

for target in targets:

    print("\n" + "=" * 60)

    print("Training:", target)

    print("=" * 60)

    model = XGBRegressor(

        n_estimators=250,

        max_depth=6,

        learning_rate=0.08,

        subsample=0.8,

        colsample_bytree=0.8,

        objective="reg:squarederror",

        tree_method="hist",

        n_jobs=-1,

        random_state=42

    )

    model.fit(

        X_train_final,

        y_train[target]

    )

    prediction = model.predict(X_test_final)

    mae = mean_absolute_error(

        y_test[target],

        prediction

    )

    rmse = np.sqrt(

        mean_squared_error(

            y_test[target],

            prediction

        )

    )

    r2 = r2_score(

        y_test[target],

        prediction

    )

    models[target] = model

    results[target] = {

        "MAE": mae,

        "RMSE": rmse,

        "R2 Score": r2

    }

    print(f"MAE      : {mae:.3f}")

    print(f"RMSE     : {rmse:.3f}")

    print(f"R2 Score : {r2:.3f}")

print("\n\nFINAL MODEL RESULTS")

results_df = pd.DataFrame(results).T

display(results_df)


Training: Service_Demand
MAE      : 4.052
RMSE     : 5.071
R2 Score : 0.734

Training: Workshop_Load_Forecast
MAE      : 4.579
RMSE     : 5.802
R2 Score : 0.881

Training: Technicians_Required
MAE      : 0.559
RMSE     : 0.707
R2 Score : 0.868

Training: Service_Bays_Required
MAE      : 0.792
RMSE     : 0.989
R2 Score : 0.714


FINAL MODEL RESULTS


,MAE,RMSE,R2 Score
Service_Demand,4.052489,5.071161,0.733583
Workshop_Load_Forecast,4.579376,5.802054,0.881092
Technicians_Required,0.559422,0.706723,0.868229
Service_Bays_Required,0.792132,0.988944,0.713636


In [5]:
# ==============================

# SAVE MODELS + PREDICTION

# ==============================

# Save trained models

joblib.dump(models, "vehicle_service_models.pkl")

joblib.dump(vectorizers, "vehicle_service_tfidf.pkl")

joblib.dump(encoder, "vehicle_service_encoder.pkl")

joblib.dump(

    {

        "text_features": text_features,

        "categorical_features": categorical_features,

        "numeric_features": numeric_features,

        "targets": targets

    },

    "vehicle_service_features.pkl"

)

print("All models and preprocessing files saved!")

# --------------------------------

# Prediction Function

# --------------------------------

def predict_service(

    customer_complaint,

    vehicle_symptoms,

    service_history,

    technician_notes,

    customer_concern,

    vehicle_type,

    vehicle_age,

    mileage,

    engine_type,

    service_category,

    previous_service_count,

    previous_repair_count,

    service_gap,

    parts_availability,

    technician_availability,

    workshop_load,

    bay_utilization,

    season,

    day_of_week,

    weekend_indicator,

    customer_segment,

    workshop_zone,

    local_demand_index,

    appointment_backlog

):

    text_values = {

        "Customer_Complaint": customer_complaint,

        "Vehicle_Symptoms": vehicle_symptoms,

        "Service_History_Text": service_history,

        "Technician_Notes": technician_notes,

        "Customer_Concern": customer_concern

    }

    # NLP transformation

    text_matrices = []

    for col in text_features:

        text = clean_text(text_values[col])

        matrix = vectorizers[col].transform([text])

        text_matrices.append(matrix)

    text_matrix = hstack(text_matrices)

    # Categorical data

    cat_data = pd.DataFrame([[

        vehicle_type,

        engine_type,

        service_category,

        season,

        day_of_week,

        customer_segment,

        workshop_zone

    ]], columns=categorical_features)

    cat_matrix = encoder.transform(cat_data)

    # Numeric data

    num_data = pd.DataFrame([[

        vehicle_age,

        mileage,

        previous_service_count,

        previous_repair_count,

        service_gap,

        parts_availability,

        technician_availability,

        workshop_load,

        bay_utilization,

        weekend_indicator,

        local_demand_index,

        appointment_backlog

    ]], columns=numeric_features)

    num_matrix = csr_matrix(num_data.values)

    # Final input

    final_input = hstack([

        text_matrix,

        cat_matrix,

        num_matrix

    ]).tocsr()

    # Predictions

    predictions = {}

    for target in targets:

        predictions[target] = models[target].predict(

            final_input

        )[0]

    return predictions

All models and preprocessing files saved!


In [6]:
# ==============================

# GRADIO INTERFACE

# ==============================

import gradio as gr

def gradio_prediction(

    customer_complaint,

    vehicle_symptoms,

    service_history,

    technician_notes,

    customer_concern,

    vehicle_type,

    vehicle_age,

    mileage,

    engine_type,

    service_category,

    previous_service_count,

    previous_repair_count,

    service_gap,

    parts_availability,

    technician_availability,

    workshop_load,

    bay_utilization,

    season,

    day_of_week,

    weekend_indicator,

    customer_segment,

    workshop_zone,

    local_demand_index,

    appointment_backlog

):

    result = predict_service(

        customer_complaint,

        vehicle_symptoms,

        service_history,

        technician_notes,

        customer_concern,

        vehicle_type,

        vehicle_age,

        mileage,

        engine_type,

        service_category,

        previous_service_count,

        previous_repair_count,

        service_gap,

        parts_availability,

        technician_availability,

        workshop_load,

        bay_utilization,

        season,

        day_of_week,

        weekend_indicator,

        customer_segment,

        workshop_zone,

        local_demand_index,

        appointment_backlog

    )

    demand = max(0, round(result["Service_Demand"]))

    load = np.clip(result["Workshop_Load_Forecast"], 0, 100)

    technicians = max(1, round(result["Technicians_Required"]))

    bays = max(1, round(result["Service_Bays_Required"]))

    # Demand level

    if demand >= 70:

        demand_level = "HIGH"

    elif demand >= 40:

        demand_level = "MEDIUM"

    else:

        demand_level = "LOW"

    # Parts requirement

    if load >= 75:

        parts_level = "HIGH"

    elif load >= 45:

        parts_level = "MEDIUM"

    else:

        parts_level = "LOW"

    recommendation = (

        f"Prepare the workshop for approximately {demand} vehicles. "

        f"Arrange {technicians} technicians and {bays} service bays. "

        f"Expected workshop load is {load:.1f}%."

    )

    return (

        demand,

        f"{load:.1f}%",

        technicians,

        bays,

        demand_level,

        parts_level,

        recommendation

    )

with gr.Blocks(

    title="Vehicle Service AI Predictor"

) as demo:

    gr.Markdown(

        """

        # 🚗 Vehicle Service AI Predictor

        ### NLP + XGBoost Workshop Resource Prediction

        Enter vehicle information and service-related text to predict

        workshop demand and resource requirements.

        """

    )

    with gr.Row():

        with gr.Column():

            gr.Markdown("### 📝 NLP Inputs")

            complaint = gr.Textbox(

                label="Customer Complaint",

                placeholder="Example: Engine making unusual noise..."

            )

            symptoms = gr.Textbox(

                label="Vehicle Symptoms",

                placeholder="Example: Vibration while accelerating..."

            )

            history = gr.Textbox(

                label="Service History",

                placeholder="Example: Previous brake repair..."

            )

            notes = gr.Textbox(

                label="Technician Notes",

                placeholder="Example: Brake pads show significant wear..."

            )

            concern = gr.Textbox(

                label="Customer Concern",

                placeholder="Example: Customer needs quick service..."

            )

        with gr.Column():

            gr.Markdown("### 🚘 Vehicle Information")

            vehicle_type = gr.Dropdown(

                ["Hatchback", "Sedan", "SUV", "MUV", "Pickup", "Van"],

                label="Vehicle Type"

            )

            vehicle_age = gr.Number(

                label="Vehicle Age (Years)",

                value=5

            )

            mileage = gr.Number(

                label="Mileage (KM)",

                value=50000

            )

            engine_type = gr.Dropdown(

                ["Petrol", "Diesel", "Electric", "Hybrid"],

                label="Engine Type"

            )

            service_category = gr.Dropdown(

                [

                    "Engine",

                    "Brake",

                    "Electrical",

                    "AC",

                    "Tyre",

                    "General",

                    "Transmission"

                ],

                label="Service Category"

            )

            previous_service_count = gr.Number(

                label="Previous Service Count",

                value=3

            )

            previous_repair_count = gr.Number(

                label="Previous Repair Count",

                value=1

            )

            service_gap = gr.Number(

                label="Average Service Gap (Days)",

                value=120

            )

    with gr.Row():

        with gr.Column():

            gr.Markdown("### 🏭 Workshop Information")

            parts = gr.Number(

                label="Parts Availability (%)",

                value=80

            )

            technicians = gr.Number(

                label="Technician Availability (%)",

                value=80

            )

            workshop_load = gr.Number(

                label="Current Workshop Load (%)",

                value=50

            )

            bay_utilization = gr.Number(

                label="Bay Utilization (%)",

                value=50

            )

            local_demand = gr.Number(

                label="Local Demand Index",

                value=50

            )

            backlog = gr.Number(

                label="Appointment Backlog",

                value=10

            )

        with gr.Column():

            gr.Markdown("### 📅 Time & Customer")

            season = gr.Dropdown(

                ["Spring", "Summer", "Monsoon", "Autumn", "Winter"],

                label="Season"

            )

            day = gr.Dropdown(

                [

                    "Monday",

                    "Tuesday",

                    "Wednesday",

                    "Thursday",

                    "Friday",

                    "Saturday",

                    "Sunday"

                ],

                label="Day of Week"

            )

            weekend = gr.Dropdown(

                [0, 1],

                label="Weekend Indicator"

            )

            customer_segment = gr.Dropdown(

                ["New", "Regular", "Premium"],

                label="Customer Segment"

            )

            workshop_zone = gr.Dropdown(

                ["Urban", "Suburban", "Rural"],

                label="Workshop Zone"

            )

    predict_btn = gr.Button(

        "🚀 Predict Service Requirements"

    )

    gr.Markdown("## 📊 Prediction Results")

    with gr.Row():

        output_demand = gr.Number(

            label="Service Demand"

        )

        output_load = gr.Textbox(

            label="Workshop Load Forecast"

        )

        output_technicians = gr.Number(

            label="Technicians Required"

        )

        output_bays = gr.Number(

            label="Service Bays Required"

        )

    with gr.Row():

        output_level = gr.Textbox(

            label="Demand Level"

        )

        output_parts = gr.Textbox(

            label="Parts Requirement"

        )

    recommendation = gr.Textbox(

        label="Workshop Recommendation",

        lines=3

    )

    predict_btn.click(

        fn=gradio_prediction,

        inputs=[

            complaint,

            symptoms,

            history,

            notes,

            concern,

            vehicle_type,

            vehicle_age,

            mileage,

            engine_type,

            service_category,

            previous_service_count,

            previous_repair_count,

            service_gap,

            parts,

            technicians,

            workshop_load,

            bay_utilization,

            season,

            day,

            weekend,

            customer_segment,

            workshop_zone,

            local_demand,

            backlog

        ],

        outputs=[

            output_demand,

            output_load,

            output_technicians,

            output_bays,

            output_level,

            output_parts,

            recommendation

        ]/

    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
